# Step 1 — Caricamento del dataset MOOC

Il dataset (MOOC User Action Dataset, Stanford SNAP) descrive le azioni degli studenti
su una piattaforma di corsi online. È diviso in 3 file TSV che descrivono le stesse
411.749 azioni da tre punti di vista diversi:

- **mooc_actions.tsv** — chi (USERID), su cosa (TARGETID), quando (TIMESTAMP)
- **mooc_action_features.tsv** — 4 caratteristiche numeriche di ogni azione (le nostre **feature X**)
- **mooc_action_labels.tsv** — la risposta giusta (la nostra **label y**): 1 = lo studente abbandona dopo questa azione, 0 = continua

La colonna `ACTIONID` è la chiave comune che collega i tre file.

In [26]:
# Importiamo pandas
import pandas as pd

DATA_DIR = "../data/act-mooc" 
# Carichiamo i 3 file del dataset MOOC.
# sep="\t" perché i file sono TSV (colonne separate da tabulazione, non da virgola)
actions  = pd.read_csv(f"{DATA_DIR}/mooc_actions.tsv", sep="\t")    # chi, cosa, quando
features = pd.read_csv(f"{DATA_DIR}/mooc_action_features.tsv", sep="\t")  # le 4 feature numeriche
labels   = pd.read_csv(f"{DATA_DIR}/mooc_action_labels.tsv", sep="\t")    # 1 = dropout, 0 = no

# .shape restituisce (righe, colonne): controlliamo che i 3 file abbiano
# lo stesso numero di righe, cioè una riga per ciascuna delle 411.749 azioni
print("actions: ", actions.shape)
print("features:", features.shape)
print("labels:  ", labels.shape)

actions:  (411749, 4)
features: (411749, 5)
labels:   (411749, 2)


## Prime righe di ogni file

`.head()` mostra le prime 5 righe: serve a "vedere" com'è fatta ogni tabella.
Nota: in un notebook solo l'ultima espressione di una cella viene visualizzata,
per questo usiamo una cella per tabella.

In [27]:
actions.head()

,ACTIONID,USERID,TARGETID,TIMESTAMP
0,0,0,0,0.0
1,1,0,1,6.0
2,2,0,2,41.0
3,3,0,1,49.0
4,4,0,2,51.0


In [28]:
features.head()

,ACTIONID,FEATURE0,FEATURE1,FEATURE2,FEATURE3
0,0,-0.319991,-0.435701,0.106784,-0.067309
1,1,-0.319991,-0.435701,0.106784,-0.067309
2,2,-0.319991,-0.435701,0.106784,-0.067309
3,3,-0.319991,-0.435701,0.106784,-0.067309
4,4,-0.319991,-0.435701,0.106784,-0.067309


In [29]:
labels.head()

,ACTIONID,LABEL
0,0,0
1,1,0
2,2,0
3,3,0
4,4,0


## Statistiche descrittive

`.describe()` calcola per ogni colonna numerica: media, deviazione standard,
minimo, massimo e quartili. Utile per un primo controllo di qualità dei dati
(es. valori impossibili o fuori scala).

In [30]:
features.describe()

,ACTIONID,FEATURE0,FEATURE1,FEATURE2,FEATURE3
count,411749.000000,4.117490e+05,4.117490e+05,4.117490e+05,4.117490e+05
mean,205874.000000,8.835428e-18,1.568289e-16,-8.614543e-17,2.084609e-17
std,118861.842332,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00
min,0.000000,-3.199915e-01,-4.357014e-01,-3.942374e-01,-6.730924e-02
25%,102937.000000,-3.199915e-01,-4.357014e-01,-3.942374e-01,-6.730924e-02
50%,205874.000000,-3.199915e-01,-4.357014e-01,1.067838e-01,-6.730924e-02
75%,308811.000000,-3.199915e-01,-4.357014e-01,1.067838e-01,-6.730924e-02
max,411748.000000,5.764755e+01,9.879679e+01,2.761694e+02,1.839709e+02


## Quanto è sbilanciato il dataset?

Contiamo quante azioni sono seguite da un abbandono (LABEL = 1) e quante no (LABEL = 0).
Questo numero è importantissimo: guiderà molte scelte del progetto
(campionamento stratificato, scelta delle metriche di valutazione, ecc.).

In [31]:
# value_counts() conta quante volte compare ogni valore nella colonna LABEL;
# con normalize=True otteniamo le percentuali invece dei conteggi assoluti
print(labels["LABEL"].value_counts())
print()
print(labels["LABEL"].value_counts(normalize=True))

LABEL
0    407683
1      4066
Name: count, dtype: int64

LABEL
0    0.990125
1    0.009875
Name: proportion, dtype: float64


## Unione dei 3 file in un'unica tabella

Uniamo i tre file tramite la chiave comune `ACTIONID`, ottenendo una sola tabella
in cui ogni riga rappresenta un'azione completa di tutte le sue informazioni:
chi l'ha fatta, quando, le sue 4 feature e la label da predire.

In [32]:
# merge() faccio combaciare le righe con lo stesso ACTIONID
# (equivale a un JOIN nel database).
# Concateniamo due merge: prima actions+features, poi il risultato+labels.
df = actions.merge(features, on="ACTIONID").merge(labels, on="ACTIONID")

print(df.shape)
df.head()

(411749, 9)


,ACTIONID,USERID,TARGETID,TIMESTAMP,FEATURE0,FEATURE1,FEATURE2,FEATURE3,LABEL
0,0,0,0,0.0,-0.319991,-0.435701,0.106784,-0.067309,0
1,1,0,1,6.0,-0.319991,-0.435701,0.106784,-0.067309,0
2,2,0,2,41.0,-0.319991,-0.435701,0.106784,-0.067309,0
3,3,0,1,49.0,-0.319991,-0.435701,0.106784,-0.067309,0
4,4,0,2,51.0,-0.319991,-0.435701,0.106784,-0.067309,0


## Conta i valori delle feature

In [33]:
for col in ["FEATURE0", "FEATURE1", "FEATURE2", "FEATURE3"]:
    print(col, "->", df[col].nunique(), "valori distinti")

FEATURE0 -> 26 valori distinti
FEATURE1 -> 16 valori distinti
FEATURE2 -> 87 valori distinti
FEATURE3 -> 164 valori distinti


## Estrazione di manuale.csv e training.csv

- **manuale.csv**: 14 campioni bilanciati (7 dropout + 7 no) perché con lo
  sbilanciamento reale (~1%) un campione casuale non conterrebbe quasi mai dropout,
  e il classificatore manuale non avrebbe esempi di entrambe le classi.
  I 14 campioni vengono inoltre scelti con vettori di feature **tutti diversi tra
  loro** (drop_duplicates): un primo tentativo con campionamento puramente casuale
  aveva prodotto campioni quasi identici, perché nel dataset molte azioni
  condividono lo stesso vettore di feature (la cella sopra mostra quanto pochi
  sono i valori distinti). Su campioni tutti uguali il classificatore manuale non
  avrebbe potuto imparare nulla.
- **training.csv**: 50.000 righe campionate in modo stratificato (proporzione 99/1
  conservata), dimensione sufficiente per l'addestramento mantenendo tempi rapidi.
  Le righe di manuale.csv vengono escluse per evitare sovrapposizioni.


In [34]:
# random_state=42 fissa il "seme" della casualità: rieseguendo il notebook
# otterremo sempre le stesse righe (riproducibilità, importante per la consegna)
SEED = 42

feature_cols = ["FEATURE0", "FEATURE1", "FEATURE2", "FEATURE3"]

# --- manuale.csv: 7 dropout + 7 non-dropout con vettori di feature distinti ---
# drop_duplicates tiene una sola riga per ogni combinazione di feature: senza
# questo passaggio il campione conteneva quasi solo righe identiche tra loro
distinti_1 = df[df["LABEL"] == 1].drop_duplicates(subset=feature_cols)
distinti_0 = df[df["LABEL"] == 0].drop_duplicates(subset=feature_cols)

dropout_si = distinti_1.sample(n=7, random_state=SEED)  # 7 righe con label 1
dropout_no = distinti_0.sample(n=7, random_state=SEED)  # 7 righe con label 0

# concateniamo le due metà e mescoliamo le righe (frac=1 = rimescola tutto)
manuale = pd.concat([dropout_si, dropout_no]).sample(frac=1, random_state=SEED)

# --- training.csv: 50.000 righe stratificate, escludendo quelle di manuale ---
resto = df.drop(manuale.index)

# groupby("LABEL") divide per classe e da ogni classe campioniamo la stessa
# frazione: la proporzione 99/1 resta identica all'originale (stratificazione)
frazione = 50_000 / len(resto)
training = resto.groupby("LABEL").sample(frac=frazione, random_state=SEED)

# --- salvataggio su file, senza la colonna indice di pandas ---
manuale.to_csv("../data/manuale.csv", index=False)
training.to_csv("../data/training.csv", index=False)

# verifica finale: dimensioni e distribuzione delle classi nei due file
print("manuale: ", manuale.shape, "-", dict(manuale["LABEL"].value_counts()))
print("training:", training.shape, "-", dict(training["LABEL"].value_counts()))

manuale:  (14, 9) - {0: np.int64(7), 1: np.int64(7)}
training: (50000, 9) - {0: np.int64(49507), 1: np.int64(493)}
